# TASK 2 — CONTEXT-AWARE FEATURE ENGINEERING
Build a fast baseline classifier to extract and learn from safe and malicious context features.

In [1]:
import pandas as pd
import numpy as np
import re
import pickle
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

print("CELL 1: Feature Extractor...")

def extract_features(prompt: str) -> dict:
    prompt_lower = prompt.lower()
    
    # Structural features
    char_count = len(prompt)
    words = prompt.split()
    word_count = len(words)
    sentence_count = prompt.count('.') + prompt.count('!') + prompt.count('?')
    sentence_count = max(1, sentence_count)
    avg_word_length = char_count / max(1, word_count)
    has_question_mark = '?' in prompt
    exclamation_count = prompt.count('!')
    
    # Lexical features
    name_pattern = r"\b(?:my name is|i am|i'm|this is)\s+([A-Z][a-z]+)\b"
    has_person_name = bool(re.search(name_pattern, prompt))
    
    code_keywords = ['def ', 'class ', 'import ', 'function', 'var ', 'const ', 'override', 'extends', 'implements']
    has_code_keywords = any(kw in prompt_lower for kw in code_keywords)
    
    has_url = bool(re.search(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', prompt))
    
    # Very basic approximation for base64 ratio
    base64_chars = len(re.findall(r'[A-Za-z0-9+/=]', prompt))
    base64_ratio = base64_chars / max(1, char_count)
    
    # Injection signal features
    injection_phrases = [
        "ignore previous", "forget your", "you are now dan",
        "system override", "no restrictions on", "do anything now",
        "bypass", "jailbreak"
    ]
    injection_phrase_count = sum(1 for phrase in injection_phrases if phrase in prompt_lower)
    
    # Safe Context Score (0-1)
    q_words = ['what', 'how', 'why', 'when', 'where']
    safe_score = 0.0
    if any(qw in prompt_lower for qw in q_words): safe_score += 0.25
    if has_code_keywords: safe_score += 0.25
    if has_person_name: safe_score += 0.25
    if "please evaluate" in prompt_lower or "analyze" in prompt_lower: safe_score += 0.25
    
    return {
        "char_count": int(char_count),
        "word_count": int(word_count),
        "sentence_count": int(sentence_count),
        "avg_word_length": float(avg_word_length),
        "has_question_mark": bool(has_question_mark),
        "exclamation_count": int(exclamation_count),
        "has_person_name": bool(has_person_name),
        "has_code_keywords": bool(has_code_keywords),
        "has_url": bool(has_url),
        "base64_ratio": float(base64_ratio),
        "injection_phrase_count": int(injection_phrase_count),
        "safe_context_score": float(safe_score)
    }

print("Sample Feature Extraction:")
sample = "Hi, my name is Dan. How do I def a function in Python?"
print(extract_features(sample))

CELL 1: Feature Extractor...
Sample Feature Extraction:
{'char_count': 54, 'word_count': 13, 'sentence_count': 2, 'avg_word_length': 4.153846153846154, 'has_question_mark': True, 'exclamation_count': 0, 'has_person_name': True, 'has_code_keywords': True, 'has_url': False, 'base64_ratio': 0.7222222222222222, 'injection_phrase_count': 0, 'safe_context_score': 0.75}


In [3]:
print("CELL 2: TF-IDF Classifier (Baseline)...")
df = pd.read_csv("exports/hard_dataset.csv").dropna(subset=['prompt'])
X = df['prompt'].tolist()
y = df['label'].tolist()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 3))
X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)

clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_val_vec)
f1 = f1_score(y_val, y_pred)
p = precision_score(y_val, y_pred)
r = recall_score(y_val, y_pred)

print(f"\n[DONE] Baseline TF-IDF Trained")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")

# Save Models
with open("exports/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
with open("exports/tfidf_classifier.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Saved exports/tfidf_classifier.pkl and exports/tfidf_vectorizer.pkl")

CELL 2: TF-IDF Classifier (Baseline)...

[DONE] Baseline TF-IDF Trained
F1 Score:  0.8837
Precision: 0.9500
Recall:    0.8261
Saved exports/tfidf_classifier.pkl and exports/tfidf_vectorizer.pkl


In [4]:
print("CELL 3: Feature Importance Analysis...")
feature_names = tfidf.get_feature_names_out()
coefs = clf.coef_[0]

# Pair up features with their weights
feat_coefs = list(zip(feature_names, coefs))

# Sort by absolute weight to find the most influential
sorted_features = sorted(feat_coefs, key=lambda x: abs(x[1]), reverse=True)

print("\nTop 50 Most Important TF-IDF Features:")
for feat, weight in sorted_features[:50]:
    direction = "MALICIOUS" if weight > 0 else "SAFE"
    print(f"{feat:<25} | Weight: {weight:>7.3f} ({direction})")

# Identify possible False Positive triggers (safe-looking ngrams with positive weight)
safe_trigger_words = ['print', 'function', 'dan', 'override', 'code', 'python', 'help', 'evaluate']
bias_alert = []
for feat, weight in feat_coefs:
    if weight > 0.5 and any(sw in feat for sw in safe_trigger_words):
        bias_alert.append(feat)

print(f"\n[!] BIAS ALERT: These safe ngrams have high malicious weight:")
print(bias_alert[:30]) # Show first 30 if there are many

CELL 3: Feature Importance Analysis...

Top 50 Most Important TF-IDF Features:
you                       | Weight:   1.808 (MALICIOUS)
the                       | Weight:   1.355 (MALICIOUS)
deutschland               | Weight:  -1.353 (SAFE)
and                       | Weight:   1.320 (MALICIOUS)
that                      | Weight:   1.309 (MALICIOUS)
sie                       | Weight:   1.118 (MALICIOUS)
europe                    | Weight:  -1.115 (SAFE)
europa                    | Weight:  -1.087 (SAFE)
will                      | Weight:   1.053 (MALICIOUS)
your                      | Weight:   1.021 (MALICIOUS)
alle                      | Weight:   1.001 (MALICIOUS)
corona                    | Weight:  -0.998 (SAFE)
all                       | Weight:   0.990 (MALICIOUS)
du                        | Weight:   0.976 (MALICIOUS)
schreibe                  | Weight:   0.963 (MALICIOUS)
germany                   | Weight:  -0.962 (SAFE)
or                        | Weight:   0.954 (MALIC